# Singapore Career Intelligence: Skills Roadmap Analysis

This notebook is an expanded version of the job analytics project, incorporating advanced data cleaning, seniority classification, and a custom **Employability Score** calculation to help jobseekers prioritize high-value skills.

## 1. Business Case & Objectives
- **Goal:** Provide evidence-based guidance for career switchers in Singapore.
- **Key Metrics:** Demand (postings volume), Reward (average salary), and Accessibility (entry-level share).
- **Success Criteria:** Identify "High-Value" skills that are not just high-paying, but also accessible to newcomers.

## 2. Setup and Data Loading
We use `duckdb` for efficient SQL-based processing and `plotly` for interactive visualizations.

In [1]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Connect to in-memory DuckDB
con = duckdb.connect(database=':memory:')

# Define file paths (adjust these to your local paths)
jobs_csv = "temp_sgjobdata_cleaned-20k.csv"
skills_csv = "temp_job_skills_cleaned-10k.csv"

## 3. Data Enrichment and Cleaning
In this step, we perform:
1. **Outlier Filtering:** Removing job postings with unrealistic salaries (below $500 or above $30,000).
2. **Seniority Classification:** Using regex to tag roles as 'Entry-level', 'Mid-level', or 'Senior-level' based on the job title.

In [2]:
con.execute(f"""
    CREATE OR REPLACE VIEW sg_jobs_raw AS 
    SELECT * FROM read_csv_auto('{jobs_csv}', all_varchar=True);
    
    CREATE OR REPLACE VIEW sg_jobs AS
    SELECT 
        categories,
        title,
        CAST(average_salary AS DOUBLE) as average_salary,
        CASE 
            WHEN lower(title) SIMILAR TO '%(intern|trainee|graduate|fresh|entry|junior)%' 
                 OR lower(positionLevels) LIKE '%fresh%' THEN 'Entry-level'
            WHEN lower(title) SIMILAR TO '%(senior|lead|principal|manager|head|director|architect)%' 
                 OR lower(positionLevels) SIMILAR TO '%(senior|manager)%' THEN 'Senior-level'
            ELSE 'Mid-level'
        END AS seniority
    FROM sg_jobs_raw
    WHERE title != 'title' 
      AND TRY_CAST(average_salary AS DOUBLE) BETWEEN 500 AND 30000;
""")

con.execute(f"CREATE OR REPLACE VIEW job_skills AS SELECT * FROM read_csv_auto('{skills_csv}')")

print("Data loaded and enriched successfully.")

Data loaded and enriched successfully.


## 4. Skills and Salary Join
We join the job postings with the skills database by matching job titles to keywords.

In [3]:
# Example filtering for 'Information Technology' - you can modify this list
selected_cat = ['Information Technology']
cats_list = str(selected_cat)

query = f"""
    SELECT 
        trim(unnest(string_split(js.job_skills, ','))) as skill,
        s.average_salary,
        s.seniority
    FROM sg_jobs s
    INNER JOIN job_skills js 
       ON lower(trim(s.title)) = lower(replace(js.job_keyword, '-', ' '))
    WHERE list_intersect(regexp_extract_all(s.categories, 'category:([^}},]+)', 1), {cats_list}::VARCHAR[]) != []
"""

raw_df = con.execute(query).df()
raw_df.head()

,skill,average_salary,seniority
0,Hiring,9000.0,Mid-level
1,Training,9000.0,Mid-level
2,Retaining,9000.0,Mid-level
3,Developing Team,9000.0,Mid-level
4,Financial Analysis,9000.0,Mid-level


## 5. The Employability Score
To rank skills effectively, we calculate a weighted score based on:
1. **Demand (50%):** Frequency of the skill in job postings.
2. **Salary (30%):** Average salary associated with the skill.
3. **Accessibility (20%):** Percentage of postings for this skill that are entry-level.

We use Min-Max scaling to normalize these metrics between 0 and 1.

In [4]:
# Group by skill to calculate stats
stats = raw_df.groupby('skill').agg(
    postings=('skill', 'count'),
    avg_salary=('average_salary', 'mean'),
    entry_count=('seniority', lambda x: (x == 'Entry-level').sum())
).reset_index()

# Filter out very low frequency skills for quality
stats = stats[stats['postings'] > 2]

# Calculate Entry-level accessibility share
stats['entry_share'] = stats['entry_count'] / stats['postings']

# Normalization (Min-Max Scaling)
for col in ['postings', 'avg_salary', 'entry_share']:
    stats[f'{col}_norm'] = (stats[col] - stats[col].min()) / (stats[col].max() - stats[col].min())

# Weighted Score Calculation
stats['employability_score'] = (
    stats['postings_norm'] * 0.5 + 
    stats['avg_salary_norm'] * 0.3 + 
    stats['entry_share_norm'] * 0.2
)

stats = stats.sort_values('employability_score', ascending=False)
stats.head(10)

,skill,postings,avg_salary,entry_count,entry_share,postings_norm,avg_salary_norm,entry_share_norm,employability_score
4702,Teamwork,1250,7709.332800,18,0.014400,1.000000,0.567473,0.021600,0.674562
3690,Project Management,1046,8577.552581,4,0.003824,0.836276,0.654995,0.005736,0.615784
2580,Java,1026,7632.796296,12,0.011696,0.820225,0.559758,0.017544,0.581548
2672,Leadership,956,8487.426778,4,0.004184,0.764045,0.645910,0.006276,0.577051
1015,Communication,926,8195.078834,8,0.008639,0.739968,0.616439,0.012959,0.557508
4079,SQL,948,7519.689873,20,0.021097,0.757624,0.548356,0.031646,0.549648
3578,Problem Solving,882,8228.668934,8,0.009070,0.704655,0.619825,0.013605,0.540996
1021,Communication skills,840,7639.747619,18,0.021429,0.670947,0.560458,0.032143,0.510040
4170,Scheduling,754,8213.179045,0,0.000000,0.601926,0.618264,0.000000,0.486442
592,Budgeting,712,8389.095506,0,0.000000,0.568218,0.635998,0.000000,0.474908


## 6. Visualizing the Results

### A. Top High-Value Skills
This chart highlights the skills with the best balance of demand, pay, and accessibility.

In [5]:
top_n = stats.head(15)
fig_score = px.bar(
    top_n, x='employability_score', y='skill', orientation='h',
    color='avg_salary', color_continuous_scale='RdYlGn',
    hover_data=['postings', 'avg_salary', 'entry_share'],
    labels={'employability_score': 'Employability Score', 'avg_salary': 'Avg Salary ($)'},
    title='Top 15 Skills by Employability Score'
)
fig_score.show()

### B. Career Pathways: Opportunity vs. Reward
This dual-axis chart shows how the number of jobs and median salary change as one progresses from entry-level to senior roles.

In [6]:
pathway_df = raw_df.groupby('seniority').agg(
    postings=('seniority', 'count'),
    median_salary=('average_salary', 'median')
).reindex(['Entry-level', 'Mid-level', 'Senior-level']).reset_index()

fig_pathway = go.Figure()

fig_pathway.add_trace(go.Bar(
    x=pathway_df['seniority'], y=pathway_df['postings'], 
    name='Number of Postings', marker_color='#2a9d8f', yaxis='y'
))

fig_pathway.add_trace(go.Scatter(
    x=pathway_df['seniority'], y=pathway_df['median_salary'], 
    name='Median Salary', line=dict(color='#e45756', width=4), yaxis='y2'
))

fig_pathway.update_layout(
    title='Career Progression: Job Volume vs. Median Salary',
    yaxis=dict(title="Volume of Jobs"),
    yaxis2=dict(title="Salary (S$)", overlaying='y', side='right'),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig_pathway.show()